# 🇻🇳 ViMind 3.0: Hệ Sinh Thái Mô Hình Ngôn Ngữ Tiếng Việt Thế Hệ Mới
### (Mixture-of-Experts 198M / Dense 64M, Qwen2.5-7B Offline Judge, Agentic RL & OpenAI API)

Notebook này được thiết kế và tối ưu hoá hoàn chỉnh cho môi trường **Kaggle GPU (Tesla T4 16GB VRAM)**:
1. **Kiến trúc MoE 198M:** 4 Chuyên gia (Experts), kích hoạt 1 chuyên gia (64M active) mỗi token với cơ chế Top-1 Routing & YaRN RoPE (32k context).
2. **Bộ dữ liệu tinh hoa ViMind 3.0 (`dataset/sft_vi_v3.jsonl`):** 52,000+ mẫu bao gồm Factual Grounding chống ảo giác (Địa lý, Lịch sử 63 tỉnh thành Việt Nam), CoT suy nghĩ `<think>`, và Sử dụng công cụ `<tool_call>`.
3. **Giám khảo Offline Qwen2.5-7B-Instruct (4-bit NF4):** Đánh giá gán nhãn cặp câu trả lời trực tiếp trên GPU không tốn phí và không bị rate-limit.
4. **Agentic RL (GRPO):** Huấn luyện mô hình tự suy luận và gọi công cụ (Toán học, Thời tiết, Thời gian) thông qua môi trường thực thi giả lập và hàm thưởng đa tiêu chí.
5. **Đóng gói & Xuất xưởng:** SafeTensors, bảng đánh giá Benchmark, và máy chủ tương thích OpenAI API.

In [ ]:
# 1. Đồng bộ mã nguồn ViMind 3.0 từ GitHub
!rm -rf /kaggle/working/vimind
!git clone https://github.com/WuKong0601/ViMind.git /kaggle/working/vimind
%cd /kaggle/working/vimind


In [ ]:
# 2. Cài đặt các gói phụ thuộc (PyTorch, Transformers, BitsAndBytes, SafeTensors, FastAPI)
!pip install -r requirements.txt
!pip install -q bitsandbytes accelerate fastapi uvicorn requests


In [ ]:
# 3. Kiểm tra phần cứng GPU Tesla T4 & môi trường CUDA
!nvidia-smi
import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU Device: {gpu_name} ({total_mem:.1f} GB VRAM)')


In [ ]:
# 4. Chạy Dry-Run kiểm thử toàn diện 5 thành phần ViMind 3.0
# (Kiểm tra MoE Routing, YaRN RoPE 32k, Loss Backward, KV Cache Generation)
!python trainer/test_3.0_dry_run.py


In [ ]:
# 5. Khởi tạo Master Dataset ViMind 3.0 (Chống ảo giác + CoT Reasoning + Tool Use)
import os
if not os.path.exists('dataset/sft_vi_v3.jsonl'):
    print('📥 Khởi tạo tập dữ liệu huấn luyện ViMind 3.0 (52,000+ mẫu)...')
    !python data_pipeline/download_distilled_3.0.py
else:
    print('⚡ Đã có sẵn dataset/sft_vi_v3.jsonl!')


In [ ]:
# 6. [GIAI ĐOẠN 1: HUẤN LUYỆN SFT 3.0 (MoE 198M / 64M Active)]
# Huấn luyện 2 epochs với MoE 4 experts và Top-1 Gating
!python -u trainer/train_sft.py \
    --data_path dataset/sft_vi_v3.jsonl \
    --tokenizer_dir model \
    --save_dir out/sft_moe \
    --save_weight vimind_3.0_moe \
    --use_moe \
    --num_experts 4 \
    --batch_size 16 \
    --accumulation_steps 4 \
    --epochs 2 \
    --learning_rate 2e-4 \
    --dtype float16 \
    --log_interval 25 \
    --save_interval 500


In [ ]:
# 7. [GIAI ĐOẠN 2: OFFLINE LLM-AS-A-JUDGE & DPO DATASET (Qwen2.5-7B 4-bit)]
# Sử dụng Qwen2.5-7B-Instruct nạp dạng 4-bit (~4.5GB VRAM) làm Giám khảo AI độc lập
# Đánh giá cặp câu trả lời và chấm điểm để tạo tập dpo_qwen_judged.jsonl
!python data_pipeline/llm_judge_dpo.py \
    --judge_model "Qwen/Qwen2.5-7B-Instruct" \
    --load_in_4bit \
    --input_candidates dataset/sft_vi_v3.jsonl \
    --output_file dataset/dpo_qwen_judged.jsonl \
    --max_samples 1500


In [ ]:
# 8. [GIAI ĐOẠN 3: AGENTIC REINFORCEMENT LEARNING (GRPO)]
# Huấn luyện khả năng sử dụng công cụ Toán học, Thời tiết, Thời gian và suy nghĩ logic
!python -u trainer/train_agent.py \
    --model_path out/sft_moe \
    --save_dir out/agent_rl \
    --save_weight vimind_3.0_agent \
    --epochs 1 \
    --num_rollouts 4 \
    --lr 1e-5 \
    --fp16


In [ ]:
# 9. [GIAI ĐOẠN 4: BENCHMARK ĐÁNH GIÁ NĂNG LỰC GỌI CÔNG CỤ (TOOL CALL)]
!python scripts/eval_toolcall.py --model_path out/agent_rl


In [ ]:
# 10. [GIAI ĐOẠN 5: ĐÓNG GÓI & XUẤT XƯỞNG HUGGING FACE SAFETENSORS]
# Chuyển đổi trọng số sang chuẩn SafeTensors, tích hợp Chat Template và Tokenizer 3.0
source_weight = 'out/agent_rl/vimind_3.0_agent.pth' if os.path.exists('out/agent_rl/vimind_3.0_agent.pth') else 'out/sft_moe/vimind_3.0_moe.pth'
!python scripts/convert_model.py \
    --input {source_weight} \
    --output /kaggle/working/vimind_3.0_moe_final \
    --moe \
    --num_experts 4


In [ ]:
# 11. [KIỂM THỬ THÀNH PHẨM VIMIND 3.0 ĐA NĂNG]
# Thử nghiệm: Factual Grounding (Hà Nội, 63 tỉnh thành), Lý luận <think>, và Gọi công cụ <tool_call>
import os, json, torch
from transformers import AutoTokenizer
from model.model import ViMindConfig, ViMindForCausalLM

model_dir = '/kaggle/working/vimind_3.0_moe_final'
if os.path.exists(model_dir):
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    config = ViMindConfig.from_pretrained(model_dir) if os.path.exists(os.path.join(model_dir, 'config.json')) else ViMindConfig(use_moe=True, num_experts=4)
    model = ViMindForCausalLM(config).cuda()
    from safetensors.torch import load_file
    if os.path.exists(os.path.join(model_dir, 'model.safetensors')):
        model.load_state_dict(load_file(os.path.join(model_dir, 'model.safetensors')), strict=False)
    model.eval()

    test_cases = [
        'Xin chào, bạn là mô hình AI nào và bạn có khả năng gì nổi bật?',
        'Thủ đô của nước Cộng hòa Xã hội Chủ nghĩa Việt Nam là gì?',
        'Việt Nam có bao nhiêu tỉnh thành? Hãy kể tên một số tỉnh miền Trung.',
        'Tính giúp tôi kết quả của 25 * 18 + 750 / 5.',
        'Thời tiết hôm nay tại Đà Nẵng thế nào, có mát mẻ không?'
    ]

    print('=' * 70)
    print('      🎉 KẾT QUẢ ĐỐI THOẠI TRỰC TIẾP VỚI VIMIND 3.0 MOE')
    print('=' * 70)
    for q in test_cases:
        prompt = tokenizer.apply_chat_template([{'role': 'user', 'content': q}], tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors='pt').input_ids.cuda()
        with torch.no_grad():
            out = model.generate(inputs, max_new_tokens=256, temperature=0.6, top_p=0.85, eos_token_id=tokenizer.eos_token_id)
        reply = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=False)
        print(f'\n👤 Người dùng: {q}')
        print(f'🤖 ViMind 3.0:\n{reply.strip()}')
        print('-' * 70)
else:
    print(f'Model directory not found at: {model_dir}')
